# Example 16 — Stokes' first problem: the impulsively started plate

At $t=0$ a wall under still fluid jumps to velocity $U$. Momentum diffuses into the fluid:
$$\frac{\partial u}{\partial t} = \nu\,\frac{\partial^2 u}{\partial y^2},\qquad u(0,t)=U,\quad u(\infty,t)=0$$
with the celebrated **self-similar** exact solution
$$\frac{u}{U} = \mathrm{erfc}\!\Big(\frac{y}{2\sqrt{\nu t}}\Big).$$

**This is the heat equation** (Example 15 Part A) wearing a velocity costume: replace
$\alpha\to\nu$, temperature → momentum. One PDE, two courses.

**One honest subtlety:** the corner $(y,t)=(0,0)$ is singular — the BC jumps
discontinuously — and a strong-form residual hates that (same story as shocks, Example 10).
We start training at $t_0 = 0.05$ with the exact profile as IC and note it openly.

Verified: L2 ≈ 4.4e-03–6.5e-03 across $t\in[0.2,1]$, ~15 s on CPU.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def g1(f, x):
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

NU, Y, T0, T1 = 1.0, 6.0, 0.05, 1.0
def u_exact(y, t): return torch.special.erfc(y/(2*torch.sqrt(NU*t)))

net = nn.Sequential(nn.Linear(2,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1)).to(device)
opt = torch.optim.Adam(net.parameters(), 2e-3)
yi = torch.linspace(0, Y, 200, device=device).reshape(-1,1)
ti0 = torch.full_like(yi, T0)

t0 = time.perf_counter()
for e in range(3000):
    opt.zero_grad()
    y = (torch.rand(2000,1,device=device)*Y).requires_grad_(True)
    t = (torch.rand(2000,1,device=device)*(T1-T0)+T0).requires_grad_(True)
    u = net(torch.cat([y,t],1))
    res = g1(u,t) - NU*g1(g1(u,y),y)
    tb = torch.rand(200,1,device=device)*(T1-T0)+T0
    loss = (res**2).mean() \
         + 10*((net(torch.cat([yi,ti0],1)) - u_exact(yi,ti0))**2).mean() \
         + 10*((net(torch.cat([torch.zeros_like(tb),tb],1)) - 1)**2).mean() \
         + 10*(net(torch.cat([torch.full_like(tb,Y),tb],1))**2).mean()
    loss.backward(); opt.step()
if device.type=='cuda': torch.cuda.synchronize()
print(f'training: {time.perf_counter()-t0:.0f} s')

# profiles AND the self-similar collapse
yg = torch.linspace(0, Y, 400, device=device).reshape(-1,1)
fig, ax = plt.subplots(1, 2, figsize=(12,4.2))
for tv, c in zip((0.1,0.3,0.6,1.0), ('tab:red','tab:orange','tab:green','tab:blue')):
    tt = torch.full_like(yg, tv)
    with torch.no_grad(): up = net(torch.cat([yg,tt],1)).cpu().numpy().ravel()
    ue = u_exact(yg, tt).cpu().numpy().ravel()
    ax[0].plot(ue, yg.cpu(), c, lw=2, alpha=.5); ax[0].plot(up, yg.cpu(), '--', color=c, lw=1.3, label=f't={tv}')
    eta = yg.cpu().numpy().ravel()/(2*np.sqrt(NU*tv))
    ax[1].plot(up, eta, '--', color=c, lw=1.2)
ax[0].set_xlabel('u/U'); ax[0].set_ylabel('y'); ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)
ax[0].set_title('Momentum diffusing from the wall (solid=erfc, dashed=PINN)')
eta = np.linspace(0, 3, 200)
from scipy.special import erfc as _erfc
ax[1].plot(_erfc(eta), eta, 'k', lw=2.2, label='erfc(η) — all times')
ax[1].set_xlabel('u/U'); ax[1].set_ylabel(r'$\eta = y/2\sqrt{\nu t}$'); ax[1].set_ylim(0,3)
ax[1].legend(fontsize=9); ax[1].grid(alpha=.3); ax[1].set_title('Self-similar collapse')
plt.tight_layout(); plt.show()

## Observations
- **Self-similarity, demonstrated:** every profile collapses onto $\mathrm{erfc}(\eta)$ when
  plotted against $\eta = y/2\sqrt{\nu t}$ — the PINN learned a family of profiles that is
  secretly one curve. (Verified L2 ≈ 6.5e-03 → 4.4e-03 from t=0.2 to 1.)
- **The singular corner is a strong-form problem.** At $(0,0)$ the data is discontinuous;
  starting at $t_0>0$ sidesteps it honestly. Compare: shocks (Ex. 10) and lid corners
  (Ex. 14) — same lesson.
- **Boundary-layer thickness** $\delta \sim 4\sqrt{\nu t}$ read straight off the plot.
- **Bridge to heat transfer:** identical to Example 15's slab — momentum and heat diffuse
  by the same mathematics (Pr = ν/α measures their ratio).

**Try:** recover ν from noisy profiles (inverse, Ex. 2); shrink $t_0$ toward 0 and watch
the corner fight back.